<div dir="rtl" align="right">

# 🏦 Bank Customer Churn Prediction (Kaggle S4E1)
**پیش‌بینی ریزش مشتریان بانک با شبکه‌های عصبی عمیق (PyTorch)**

خوش آمدید! 🚀 در این نوت‌بوک، یک پایپ‌لاین کامل یادگیری عمیق برای پیش‌بینی احتمال ترک یا ماندن مشتریان در بانک طراحی شده است[cite: 1]. این پروژه بر اساس داده‌های مسابقه **Kaggle Playground Series - Season 4, Episode 1** توسعه یافته و معیار ارزیابی نهایی آن **ROC-AUC** می‌باشد[cite: 1].

---

## 📋 نقشه راه و مراحل پروژه (Pipeline Overview)

### ۱. 🛠️ مهندسی ویژگی‌ها (Feature Engineering)
برای کمک به درک بهتر الگوهای مالی توسط شبکه عصبی، علاوه بر ویژگی‌های خام، چند ویژگی ترکیبی جدید مهندسی و به داده‌ها اضافه شده است[cite: 1]:
* `balance_to_salary`: نسبت موجودی حساب به حقوق تخمینی مشتری[cite: 1].
* `tenure_to_age`: نسبت سابقه حضور مشتری در بانک به سن او[cite: 1].
* `balance_per_product`: میانگین موجودی حساب به ازای هر محصول بانکی که مشتری در اختیار دارد[cite: 1].

### ۲. 🧹 پیش‌پردازش داده‌ها (Data Preprocessing)
* استفاده از **One-Hot Encoding** برای متغیرهای دسته‌ای و غیرخطی (مثل موقعیت جغرافیایی و تعداد محصولات)[cite: 1].
* استانداردسازی ویژگی‌های عددی با **StandardScaler** برای تسریع همگرایی و جلوگیری از تسلط ویژگی‌های با مقیاس بزرگ[cite: 1].
* تبدیل فریم‌های داده به تانسورهای استاندارد `PyTorch` (`torch.float32`) جهت پردازش روی GPU[cite: 1].

### ۳. 🧠 معماری مدل‌ها (Model Architectures)
در این پروژه، به جای تکیه بر یک مدل واحد، **۴ معماری مختلف از شبکه‌های عصبی چندلایه (MLP)** طراحی و ارزیابی شده‌اند تا نقاط ضعف یکدیگر را پوشش دهند[cite: 1]:
* **Model 1 (37K Params):** معماری عمیق با لایه‌های `LayerNorm` و تابع فعال‌ساز `ELU` (ثبت بهترین عملکرد تکی با امتیاز ~0.888)[cite: 1].
* **Model 2 (41K Params):** معماری کلاسیک و ساده‌تر بدون استفاده از لایه‌های نرمال‌سازی[cite: 1].
* **Model 3 (326K Params):** معماری عریض (تا ۵۱۲ نورون) مجهز به `BatchNorm1d`، تابع فعال‌ساز مدرن `GELU` و لایه‌های `Dropout` برای جلوگیری از بیش‌برازش[cite: 1].
* **Model 4 (43K Params):** معماری پایه با تابع `ELU`[cite: 1].

### ۴. 🏋️‍♂️ حلقه آموزش و زمان‌بندی (Training & Validation Loop)
* **تابع زیان:** استفاده از `BCEWithLogitsLoss` برای پایداری عددی بالاتر در دسته‌بندی باینری[cite: 1].
* **بهینه‌ساز:** `AdamW` با اعمال نرخ `weight_decay` جهت منظم‌سازی (Regularization)[cite: 1].
* **زمان‌بندی نرخ یادگیری:** استفاده از زمان‌بند واکنشی `ReduceLROnPlateau` برای کاهش خودکار Learning Rate در زمان توقف پیشرفت مدل[cite: 1].
* ذخیره خودکار بهترین وزن‌ها (Best Checkpoints) در طول ۶۰ اپوک ارزیابی[cite: 1].

### ۵. 🤝 ترکیب مدل‌ها (Ensembling & Blending)
در نهایت برای پرش از محدودیت‌های شبکه‌های تکی و رسیدن به بالاترین امتیاز ممکن (کاهش واریانس خطای پیش‌بینی)، خروجی احتمالات مدل‌های برتر با استراتژی **Weighted Blending** ترکیب شده‌اند که امتیاز ROC-AUC را با موفقیت ارتقا داده است[cite: 1].

---
**🛠️ ابزارهای استفاده شده:** `PyTorch` | `Pandas` | `Scikit-learn` | `KaggleHub`

</div>

### Imports

In [56]:
import os
import kagglehub
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset , DataLoader

In [57]:
print("s")

s


### Download data

In [58]:
# !mkdir -p ~/.kaggle && echo KGAT_10f6ee12d4bece7f3843176f24735bff > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [59]:
# Download latest version
# path = kagglehub.competition_download('playground-series-s4e1')

# print("Path to competition files:", path)

In [60]:
# print(os.listdir(path))

## creat data frame

In [61]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv("test.csv")

In [62]:
df_train.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97,0
1,1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50,0
2,2,15694510,Hsueh,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69,0
3,3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0
4,4,15766172,Chiemenam,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83,0


In [63]:
df_train.shape

(165034, 14)

In [64]:
df_test.shape

(110023, 13)

In [65]:
X_train = df_train.drop(["id","CustomerId","Surname","Exited"],axis = 1)
X_test = df_test.drop(["id","CustomerId","Surname"],axis = 1)

y_train = df_train["Exited"]


### Feature Engineering

In [66]:
X_test["balance_to_salary"] = X_test["Balance"] / X_test["EstimatedSalary"]
X_train["balance_to_salary"] = X_train["Balance"] / X_train["EstimatedSalary"]

In [67]:
X_train["tenure_to_age"]  = X_train["Tenure"] / X_train["Age"]
X_test["tenure_to_age"]  = X_test["Tenure"] / X_test["Age"]

In [68]:
X_train["balance_per_product"] = X_train["Balance"] / X_train["NumOfProducts"]
X_test["balance_per_product"] = X_test["Balance"] / X_test["NumOfProducts"]

## clean and Scale

In [69]:
X_train.dtypes

CreditScore              int64
Geography                  str
Gender                     str
Age                    float64
Tenure                   int64
Balance                float64
NumOfProducts            int64
HasCrCard              float64
IsActiveMember         float64
EstimatedSalary        float64
balance_to_salary      float64
tenure_to_age          float64
balance_per_product    float64
dtype: object

In [70]:
X_train[["HasCrCard","IsActiveMember"]] = X_train[["HasCrCard","IsActiveMember"]].astype("int64")
X_test[["HasCrCard","IsActiveMember"]] = X_test[["HasCrCard","IsActiveMember"]].astype("int64")


In [71]:
X_train["Gender"] = X_train["Gender"].map({"Male" : 1,"Female":0})
X_test["Gender"] = X_test["Gender"].map({"Male" : 1,"Female":0})

In [72]:
X_train_trn , X_train_val , y_train_trn , y_train_val = train_test_split(X_train ,y_train , test_size= .2 , random_state= 42 ,stratify=y_train)

In [73]:
onehot_encoder = OneHotEncoder(handle_unknown= "ignore",sparse_output= False)
onehot_encoder.set_output(transform="pandas")

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_cate

In [74]:
X_train_trn_onehot = onehot_encoder.fit_transform(X_train_trn[["Geography","NumOfProducts"]])
X_train_val_onehot = onehot_encoder.transform(X_train_val[['Geography',"NumOfProducts"]])
X_test_onehot = onehot_encoder.transform(X_test[['Geography',"NumOfProducts"]])

In [75]:
X_train_trn = X_train_trn.drop(columns=["Geography","NumOfProducts"])
X_train_trn = pd.concat([X_train_trn,X_train_trn_onehot],axis = 1)

X_train_val = X_train_val.drop(columns=["Geography","NumOfProducts"])
X_train_val = pd.concat([X_train_val,X_train_val_onehot],axis = 1)

X_test = X_test.drop(columns=["Geography","NumOfProducts"])
X_test = pd.concat([X_test,X_test_onehot],axis = 1)



In [76]:
list_of_num = ["CreditScore","Age","Tenure","Balance","EstimatedSalary","balance_to_salary","tenure_to_age","balance_per_product"]
scaler = StandardScaler()

In [77]:
X_train_trn[list_of_num] = scaler.fit_transform(X_train_trn[list_of_num])
X_train_val[list_of_num] = scaler.transform(X_train_val[list_of_num])
X_test[list_of_num] = scaler.transform(X_test[list_of_num])


In [78]:
print(f"X_train_trn = {X_train_trn.shape} , X_train_val = {X_train_val.shape} , X_test = {X_test.shape}")

X_train_trn = (132027, 18) , X_train_val = (33007, 18) , X_test = (110023, 18)


In [79]:
X_train_trn.dtypes

CreditScore            float64
Gender                   int64
Age                    float64
Tenure                 float64
Balance                float64
HasCrCard                int64
IsActiveMember           int64
EstimatedSalary        float64
balance_to_salary      float64
tenure_to_age          float64
balance_per_product    float64
Geography_France       float64
Geography_Germany      float64
Geography_Spain        float64
NumOfProducts_1        float64
NumOfProducts_2        float64
NumOfProducts_3        float64
NumOfProducts_4        float64
dtype: object

### array to tensor

In [80]:
y_train_trn = pd.DataFrame(y_train_trn)
y_train_val = pd.DataFrame(y_train_val)

In [81]:
X_train_trn_tensor = torch.tensor(X_train_trn.values, dtype=torch.float32)
X_train_val_tensor = torch.tensor(X_train_val.values,dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values,dtype=torch.float32)

y_train_trn_tensor = torch.tensor(y_train_trn.values,dtype=torch.float32)
y_train_val_tensor = torch.tensor(y_train_val.values,dtype=torch.float32)

In [82]:
y_train_val_tensor.shape

torch.Size([33007, 1])

## create model

model

In [83]:
input_dim = X_train_trn.shape[1]
#parameters = 36,225
model_1 = nn.Sequential(
    nn.Linear(input_dim,64),
    nn.LayerNorm(64),
    nn.ELU(),
    nn.Linear(64,128),
    nn.LayerNorm(128),
    nn.ELU(),
    nn.Linear(128,128),
    nn.LayerNorm(128),
    nn.ELU(),
    nn.Linear(128,64),
    nn.LayerNorm(64),
    nn.ELU(),
    nn.Linear(64,32),
    nn.LayerNorm(32),
    nn.ELU(),
    nn.Linear(32,1)
)
#parametes = 41,313
model_2 = nn.Sequential(
    nn.Linear(input_dim,256),
    nn.ELU(),
    nn.Linear(256,128),
    nn.ELU(),
    nn.Linear(128,32),
    nn.ELU(),
    nn.Linear(32,1)
)
#parametes = 69,929
model_3 = nn.Sequential(
    nn.Linear(input_dim,64),
    nn.BatchNorm1d(64),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(64,128),
    nn.BatchNorm1d(128),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(128,256),
    nn.BatchNorm1d(256),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(256,512),
    nn.BatchNorm1d(512),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(512,256),
    nn.BatchNorm1d(256),
    nn.GELU(),
    nn.Dropout(0.15),
    nn.Linear(256,64),
    nn.BatchNorm1d(64),
    nn.GELU(),
    nn.Dropout(0.10),
    nn.Linear(64,32),
    nn.BatchNorm1d(32),
    nn.GELU(),
    nn.Linear(32,1)
)

#parametes = 43,329
model_4 = nn.Sequential(
    nn.Linear(input_dim,128),
    nn.ELU(),
    nn.Linear(128,256),
    nn.ELU(),
    nn.Linear(256,32),
    nn.ELU(),
    nn.Linear(32,1)
)


In [108]:

i = 1
for model in [model_1,model_2,model_3,model_4]:
    total_params = 0
    for p in model.parameters():
        if p.requires_grad:
            total_params += p.numel()

    print(f"تعداد کل پارامترهای = {i}", total_params)
    i +=1

تعداد کل پارامترهای = 1 37249
تعداد کل پارامترهای = 2 41921
تعداد کل پارامترهای = 3 326657
تعداد کل پارامترهای = 4 43713


### loss and optimazer

In [94]:
criterion = nn.BCEWithLogitsLoss()

optimazer_1 = torch.optim.AdamW(model_1.parameters(),lr=0.001,weight_decay=0.0001)
optimazer_2 = torch.optim.AdamW(model_2.parameters(),lr=0.001,weight_decay=0.0001)
optimazer_3 = torch.optim.AdamW(model_3.parameters(),lr=0.001,weight_decay=0.0001)
optimazer_4 = torch.optim.AdamW(model_4.parameters(),lr=0.001,weight_decay=0.0001)


In [95]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimazer_1, mode='min', factor=0.5, patience=3
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimazer_2, mode='min', factor=0.5, patience=3
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimazer_3, mode='min', factor=0.5, patience=3
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimazer_4, mode='min', factor=0.5, patience=3
)

## Train loop

In [96]:
train_detaset = TensorDataset(X_train_trn_tensor, y_train_trn_tensor)
val_dataset = TensorDataset(X_train_val_tensor , y_train_val_tensor)

In [97]:
train_loader = DataLoader(train_detaset ,batch_size = 256 , shuffle= True)
val_loader = DataLoader(val_dataset ,batch_size = 256 , shuffle= False)

In [98]:
print(torch.cuda.is_available())

True


In [103]:
val_losses_1 = []
train_losses_1 = []

current_loss_1 = float("inf")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_1 = model_1.to(device)


num_epochs = 60

for epoch in range(num_epochs):

    #zero grad
    model_1.train()
    train_loss = 0.0

    for x_batch ,y_batch in train_loader:

        X_batch,y_batch = x_batch.to(device) , y_batch.to(device)

        optimazer_1.zero_grad()

        #forwardpass
        output_model_1 = model_1(X_batch)
        loss_1 = criterion(output_model_1,y_batch)


        #backward
        loss_1.backward()
        optimazer_1.step()

        train_loss += loss_1.item() * X_batch.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)
    train_losses_1.append(avg_train_loss)
    #Validation Phase
    model_1.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                y_val_pred = model_1(X_batch)
                loss = criterion(y_val_pred, y_batch)

                val_loss += loss.item() * X_batch.size(0)

    avg_val_loss = val_loss / len(val_loader.dataset)

    scheduler.step(avg_val_loss)
    val_losses_1.append(avg_val_loss)

    if avg_val_loss < current_loss_1:
        current_loss_1 = avg_val_loss
        torch.save(model_1.state_dict(), "best_model_1.pth")
        print("_____________________________________________________________> model_1")

    model_1.load_state_dict(torch.load("best_model_1.pth"))
    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

# بازگردانی بهترین وزن‌ها پس از اتمام آموزش
model_1.load_state_dict(torch.load("best_model_1.pth"))

_____________________________________________________________> model_1
Epoch [01/60] | Train Loss: 0.3189 | Val Loss: 0.3226
Epoch [02/60] | Train Loss: 0.3191 | Val Loss: 0.3239
Epoch [03/60] | Train Loss: 0.3190 | Val Loss: 0.3228
Epoch [04/60] | Train Loss: 0.3188 | Val Loss: 0.3232
Epoch [05/60] | Train Loss: 0.3189 | Val Loss: 0.3235
Epoch [06/60] | Train Loss: 0.3188 | Val Loss: 0.3236
_____________________________________________________________> model_1
Epoch [07/60] | Train Loss: 0.3192 | Val Loss: 0.3224
Epoch [08/60] | Train Loss: 0.3186 | Val Loss: 0.3247
_____________________________________________________________> model_1
Epoch [09/60] | Train Loss: 0.3191 | Val Loss: 0.3218
Epoch [10/60] | Train Loss: 0.3187 | Val Loss: 0.3237
Epoch [11/60] | Train Loss: 0.3188 | Val Loss: 0.3238
Epoch [12/60] | Train Loss: 0.3188 | Val Loss: 0.3228
Epoch [13/60] | Train Loss: 0.3187 | Val Loss: 0.3224
Epoch [14/60] | Train Loss: 0.3188 | Val Loss: 0.3231
Epoch [15/60] | Train Loss: 0.3

<All keys matched successfully>

In [104]:
val_losses_2 = []
train_losses_2 = []

current_loss_2 = float("inf")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_2 = model_2.to(device)

num_epochs = 60

for epoch in range(num_epochs):

    model_2.train()
    train_loss = 0.0

    for x_batch, y_batch in train_loader:

        X_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimazer_2.zero_grad()

        output_model_2 = model_2(X_batch)
        loss_2 = criterion(output_model_2, y_batch)

        loss_2.backward()
        optimazer_2.step()

        train_loss += loss_2.item() * X_batch.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)
    train_losses_2.append(avg_train_loss)

    model_2.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            y_val_pred = model_2(X_batch)
            loss = criterion(y_val_pred, y_batch)

            val_loss += loss.item() * X_batch.size(0)

    avg_val_loss = val_loss / len(val_loader.dataset)

    scheduler.step(avg_val_loss)
    val_losses_2.append(avg_val_loss)

    if avg_val_loss < current_loss_2:
        current_loss_2 = avg_val_loss
        torch.save(model_2.state_dict(), "best_model_2.pth")
        print("_____________________________________________________________> model_2")

    model_2.load_state_dict(torch.load("best_model_2.pth"))
    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

# بازگردانی بهترین وزن‌ها پس از اتمام آموزش
model_2.load_state_dict(torch.load("best_model_2.pth"))

_____________________________________________________________> model_2
Epoch [01/60] | Train Loss: 0.3408 | Val Loss: 0.3289
_____________________________________________________________> model_2
Epoch [02/60] | Train Loss: 0.3270 | Val Loss: 0.3278
_____________________________________________________________> model_2
Epoch [03/60] | Train Loss: 0.3250 | Val Loss: 0.3258
_____________________________________________________________> model_2
Epoch [04/60] | Train Loss: 0.3242 | Val Loss: 0.3251
Epoch [05/60] | Train Loss: 0.3237 | Val Loss: 0.3260
_____________________________________________________________> model_2
Epoch [06/60] | Train Loss: 0.3237 | Val Loss: 0.3235
Epoch [07/60] | Train Loss: 0.3230 | Val Loss: 0.3241
Epoch [08/60] | Train Loss: 0.3230 | Val Loss: 0.3235
Epoch [09/60] | Train Loss: 0.3231 | Val Loss: 0.3236
Epoch [10/60] | Train Loss: 0.3232 | Val Loss: 0.3239
Epoch [11/60] | Train Loss: 0.3233 | Val Loss: 0.3236
Epoch [12/60] | Train Loss: 0.3234 | Val Loss: 0.32

<All keys matched successfully>

In [105]:
val_losses_3 = []
train_losses_3 = []

current_loss_3 = float("inf")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_3 = model_3.to(device)

num_epochs = 60

for epoch in range(num_epochs):

    model_3.train()
    train_loss = 0.0

    for x_batch, y_batch in train_loader:
        X_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimazer_3.zero_grad()

        # forwardpass
        output_model_3 = model_3(X_batch)
        loss_3 = criterion(output_model_3, y_batch)

        # backward
        loss_3.backward()
        optimazer_3.step()

        train_loss += loss_3.item() * X_batch.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)
    train_losses_3.append(avg_train_loss)

    model_3.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            y_val_pred = model_3(X_batch)
            loss = criterion(y_val_pred, y_batch)

            val_loss += loss.item() * X_batch.size(0)

    avg_val_loss = val_loss / len(val_loader.dataset)
    scheduler.step(avg_val_loss)
    val_losses_3.append(avg_val_loss)

    if avg_val_loss < current_loss_3:
        current_loss_3 = avg_val_loss
        torch.save(model_3.state_dict(), "best_model_3.pth")
        print("_____________________________________________________________> model_3")

    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

# بازگردانی بهترین وزن‌ها پس از اتمام آموزش
model_3.load_state_dict(torch.load("best_model_3.pth"))

_____________________________________________________________> model_3
Epoch [01/60] | Train Loss: 0.3687 | Val Loss: 0.3267
_____________________________________________________________> model_3
Epoch [02/60] | Train Loss: 0.3341 | Val Loss: 0.3258
_____________________________________________________________> model_3
Epoch [03/60] | Train Loss: 0.3311 | Val Loss: 0.3254
_____________________________________________________________> model_3
Epoch [04/60] | Train Loss: 0.3298 | Val Loss: 0.3253
_____________________________________________________________> model_3
Epoch [05/60] | Train Loss: 0.3284 | Val Loss: 0.3237
_____________________________________________________________> model_3
Epoch [06/60] | Train Loss: 0.3280 | Val Loss: 0.3235
Epoch [07/60] | Train Loss: 0.3280 | Val Loss: 0.3240
_____________________________________________________________> model_3
Epoch [08/60] | Train Loss: 0.3270 | Val Loss: 0.3230
Epoch [09/60] | Train Loss: 0.3271 | Val Loss: 0.3241
_________________

<All keys matched successfully>

In [106]:
val_losses_4 = []
train_losses_4 = []

current_loss_4 = float("inf")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_4 = model_4.to(device)

num_epochs = 60

for epoch in range(num_epochs):

    # ----------------- فاز آموزش -----------------
    model_4.train()
    train_loss = 0.0

    for x_batch, y_batch in train_loader:
        X_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimazer_4.zero_grad()

        # forwardpass
        output_model_4 = model_4(X_batch)
        loss_4 = criterion(output_model_4, y_batch)

        # backward
        loss_4.backward()
        optimazer_4.step()

        train_loss += loss_4.item() * X_batch.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)
    train_losses_4.append(avg_train_loss)

    model_4.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            y_val_pred = model_4(X_batch)
            loss = criterion(y_val_pred, y_batch)

            val_loss += loss.item() * X_batch.size(0)

    avg_val_loss = val_loss / len(val_loader.dataset)
    scheduler.step(avg_val_loss)
    val_losses_4.append(avg_val_loss)

    if avg_val_loss < current_loss_4:
        current_loss_4 = avg_val_loss
        torch.save(model_4.state_dict(), "best_model_4.pth")
        print("_____________________________________________________________> model_4")

    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

# بازگردانی بهترین وزن‌ها پس از اتمام آموزش
model_4.load_state_dict(torch.load("best_model_4.pth"))

_____________________________________________________________> model_4
Epoch [01/60] | Train Loss: 0.7244 | Val Loss: 0.7242
_____________________________________________________________> model_4
Epoch [02/60] | Train Loss: 0.7239 | Val Loss: 0.7238
_____________________________________________________________> model_4
Epoch [03/60] | Train Loss: 0.7235 | Val Loss: 0.7234
_____________________________________________________________> model_4
Epoch [04/60] | Train Loss: 0.7231 | Val Loss: 0.7229
_____________________________________________________________> model_4
Epoch [05/60] | Train Loss: 0.7227 | Val Loss: 0.7225
_____________________________________________________________> model_4
Epoch [06/60] | Train Loss: 0.7222 | Val Loss: 0.7221
_____________________________________________________________> model_4
Epoch [07/60] | Train Loss: 0.7218 | Val Loss: 0.7217
_____________________________________________________________> model_4
Epoch [08/60] | Train Loss: 0.7214 | Val Loss: 0.7212


KeyboardInterrupt: 

In [107]:
from sklearn.metrics import roc_auc_score

def evaluate_auc(model, val_loader, device):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            # تبدیل خروجی خام به احتمال بین ۰ و ۱ با سیگموئید
            logits = model(X_batch)
            probs = torch.sigmoid(logits)
            all_preds.extend(probs.cpu().numpy())
            all_targets.extend(y_batch.numpy())

    return roc_auc_score(all_targets, all_preds)

# print(f"Model 1 Validation ROC-AUC: {evaluate_auc(model_1, val_loader, device):.5f}")
print(f"Model 1 Validation ROC-AUC: {evaluate_auc(model_1, val_loader, device):.5f}")
print(f"Model 2 Validation ROC-AUC: {evaluate_auc(model_2, val_loader, device):.5f}")
print(f"Model 3 Validation ROC-AUC: {evaluate_auc(model_3, val_loader, device):.5f}")
print(f"Model 4 Validation ROC-AUC: {evaluate_auc(model_4, val_loader, device):.5f}")


Model 1 Validation ROC-AUC: 0.88815
Model 2 Validation ROC-AUC: 0.88723
Model 3 Validation ROC-AUC: 0.88814
Model 4 Validation ROC-AUC: 0.36084


Model 3 Validation ROC-AUC: 0.88200

In [109]:
from sklearn.metrics import roc_auc_score

model_1.eval()
model_2.eval()
model_3.eval()

preds_1, preds_2, preds_3 = [], [], []
y_true = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)

        # محاسبه احتمالات هر سه مدل
        p1 = torch.sigmoid(model_1(X_batch)).cpu().numpy()
        p2 = torch.sigmoid(model_2(X_batch)).cpu().numpy()
        p3 = torch.sigmoid(model_3(X_batch)).cpu().numpy()

        preds_1.extend(p1)
        preds_2.extend(p2)
        preds_3.extend(p3)
        y_true.extend(y_batch.numpy())

# تبدیل به آرایه
preds_1 = np.array(preds_1).ravel()
preds_2 = np.array(preds_2).ravel()
preds_3 = np.array(preds_3).ravel()
y_true = np.array(y_true).ravel()

# میانگین احتمالات ۳ مدل سالم
ensemble_val_preds = (preds_1 + preds_2 + preds_3) / 3

ensemble_auc = roc_auc_score(y_true, ensemble_val_preds)
print(f"Ensemble (Model 1 + 2 + 3) Validation ROC-AUC: {ensemble_auc:.5f}")

Ensemble (Model 1 + 2 + 3) Validation ROC-AUC: 0.88845


In [110]:
# میانگین دو مدل که عملکرد بهتری داشتند
blend_1_3 = (preds_1 + preds_3) / 2
print(f"Blend (Model 1 + 3) ROC-AUC: {roc_auc_score(y_true, blend_1_3):.5f}")

Blend (Model 1 + 3) ROC-AUC: 0.88859


In [111]:
# وزن‌دهی بر اساس قدرت مدل‌ها
weighted_blend = (0.45 * preds_1) + (0.10 * preds_2) + (0.45 * preds_3)
print(f"Weighted Blend ROC-AUC: {roc_auc_score(y_true, weighted_blend):.5f}")

Weighted Blend ROC-AUC: 0.88858


In [ ]:
# ۱. قرار دادن مدل در حالت ارزیابی
model_1.eval()

# ۲. پیش‌بینی روی داده‌های تست بدون محاسبه گرادیان
with torch.no_grad():
    # انتقال داده‌های تست به همان دستگاه مدل (CPU یا GPU)
    X_test_device = X_test_tensor.to(device)

    # دریافت خروجی خام مدل (Logits)
    test_logits = model_1(X_test_device)

    # تبدیل به احتمال بین ۰ و ۱ با تابع سیگموئید و تبدیل به آرایه ۱ بعدی نامپای
    test_preds = torch.sigmoid(test_logits).cpu().numpy().squeeze()

# ۳. ساخت دیتافریم خروجی مطابق فرمت کگل
submission = pd.DataFrame({
    "id": df_test["id"],
    "Exited": test_preds
})

# ۴. ذخیره فایل CSV بدون ستون ایندکس اضافه
submission.to_csv("submission_01.csv", index=False)

print("فایل submission.csv با موفقیت ساخته شد!")
submission.head(10)